In [307]:
import pandas as pd
from pathlib import Path
import os

In [308]:
# REPO_ROOT = Path(__file__).resolve().parents[1]
# RAW_DATA_DIR = REPO_ROOT / 'data' / 'raw'

In [309]:
RAW_DATA_DIR = "../data/raw"

In [310]:
#cell_author_info = pd.read_csv(RAW_DATA_DIR + '/' + 'Cell_2000-2026_author_info.csv')
cell_author_info = pd.read_csv(RAW_DATA_DIR + '/' + 'Nature_2000-2026_author_info.csv')

In [311]:
cell_author_info.head()

,Unnamed: 0,PMID,author,affiliations
0,0,11241975,Martin Bobrow,Martin Bobrow is in the Department of Medical ...
1,1,11241975,Sandy Thomas,No affiliation
2,2,11185494,T Lincoln,No affiliation
3,3,10809627,D Dickson,No affiliation
4,4,10809626,N Loder,No affiliation


In [312]:
# drop one column and set PMID to index
cell_author_info = cell_author_info.drop(columns = ['Unnamed: 0']).set_index('PMID')

In [313]:
cell_author_info.head()

,author,affiliations
PMID,,
11241975,Martin Bobrow,Martin Bobrow is in the Department of Medical ...
11241975,Sandy Thomas,No affiliation
11185494,T Lincoln,No affiliation
10809627,D Dickson,No affiliation
10809626,N Loder,No affiliation


In [314]:
# drop authors without an affiliation listed
# cond = cell_author_info['affiliations'] != 'No affiliation'
# cell_author_info = cell_author_info[cond]


In [315]:
cell_author_info.tail(10)

,author,affiliations
PMID,,
40770093,Iwona Czaban,Division of Biological and Environmental Scien...
40770093,Mariusz Jaremko,Division of Biological and Environmental Scien...
40770093,Michael-Christopher Keogh,"EpiCypher, Durham, NC, USA."
40770093,Kang Le,"Therapeutics Discovery Division, The Universit..."
40770093,Michael J Soth,"Therapeutics Discovery Division, The Universit..."
40770093,Benjamin A Garcia,Department of Biochemistry and Molecular Bioph...
40770093,Łukasz Jaremko,Division of Biological and Environmental Scien...
40770093,Jacek Majewski,"Department of Human Genetics, McGill Universit..."
40770093,Pawel K Mazur,"Department of Experimental Radiation Oncology,..."


In [316]:
# split affiliations
# NOTE: affiliations in Cell appear to be split by semi-colons and not separated in the same way as the Nature affiliations 
cell_author_info['affiliations'] = cell_author_info['affiliations'].str.split('|')

In [317]:
# set primary affiliation as the first affiliation to appear in the affiliation list
cell_author_info['primary_affiliation'] = [affil[0] for affil in cell_author_info['affiliations']]

In [318]:
cell_author_info.tail(10)['primary_affiliation'].iloc[0]

'Division of Biological and Environmental Science and Engineering, King Abdullah University of Science and Technology, Thuwal, Saudi Arabia.'

In [319]:
import re
# email_pat = re.compile(r'Electronic address: (.*)')
# match = re.search(email_pat, cell_author_info['primary_affiliation'].iloc[-3])
# match.group(1)

In [320]:
cell_author_info['primary_affiliation'].iloc[-3]

'Department of Human Genetics, McGill University, Montreal, Quebec, Canada. jacek.majewski@mcgill.ca.'

In [321]:
EMAIL_PAT = re.compile(r'\.?\s*(?:electronic address:|email:|e-mail:)?\s*[\w\.-]+@[\w\.-]+\.\w+\.?$', flags=re.IGNORECASE)

# build dictionary of common alternate forms of different countries that show up in the data
koreas = ['Republic of Korea', 'Korea', 'South Korea']
netherlands = ['the Netherlands', 'The Netherlands', 'Netherlands']
china = ['China', 'P.R. China', "People's Republic of China", "PRC", 'Fujian', 'Chinese Academy of Sciences', 'Hefei Comprehensive National Science Center']
uk = ['UK', 'United Kingdom', 'Oxford UK OX BN']
usa = ['Howard Hughes Medical Institute', 'MA', 'CA', 'NY', 'PA', 'MD', 'Massachusetts', 'NC', 'Texas', 'Maryland', 'CA USA', 'TX -', 'CT', 'CA -' 'http://www.jcsg.org']
mexico = ['México', 'Mexico']

COUNTRY_MAP = {}
for item in koreas:
    COUNTRY_MAP[item] = 'South Korea'
for item in netherlands:
    COUNTRY_MAP[item] = 'the Netherlands'
for item in china:
    COUNTRY_MAP[item] = 'China'
for item in uk:
    COUNTRY_MAP[item] = 'UK'
for item in usa:
    COUNTRY_MAP[item] = 'USA'
for item in mexico:
    COUNTRY_MAP[item] = 'Mexico'


def extract_country(affiliation:str) -> str:
    remove_email_text = re.sub(EMAIL_PAT, '', affiliation)
    cleaned = remove_email_text.strip().rstrip('.')
    country = cleaned.split(',')[-1].strip()
    country = re.sub(r'\d+', '', country).strip()
    country = COUNTRY_MAP.get(country, country)
    
    return country

In [322]:
extract_country(cell_author_info['primary_affiliation'].iloc[-1])

'USA'

In [323]:
cell_author_info['primary_country'] = cell_author_info['primary_affiliation'].apply(extract_country)

In [324]:
cell_author_info.tail()

,author,affiliations,primary_affiliation,primary_country
PMID,,,,
40770093,Benjamin A Garcia,[Department of Biochemistry and Molecular Biop...,Department of Biochemistry and Molecular Bioph...,USA
40770093,Łukasz Jaremko,[Division of Biological and Environmental Scie...,Division of Biological and Environmental Scien...,Saudi Arabia
40770093,Jacek Majewski,"[Department of Human Genetics, McGill Universi...","Department of Human Genetics, McGill Universit...",Canada
40770093,Pawel K Mazur,[Department of Experimental Radiation Oncology...,"Department of Experimental Radiation Oncology,...",USA
40770093,Or Gozani,"[Department of Biology, Stanford University, S...","Department of Biology, Stanford University, St...",USA


In [325]:
cell_author_info['primary_country'].value_counts().describe()

count       934.000000
mean        391.141328
std        5375.288616
min           1.000000
25%           1.000000
50%           1.000000
75%           5.000000
max      115524.000000
Name: count, dtype: float64

In [326]:
cell_author_info = cell_author_info[cell_author_info['primary_country'].map(cell_author_info['primary_country'].value_counts()) > 10]

In [333]:
cell_author_info.tail(50)

,author,affiliations,primary_affiliation,primary_country
PMID,,,,
40804525,Omar Abdulla Omar,"[Ledi-Geraru Research Project Field Team, Mill...","Ledi-Geraru Research Project Field Team, Mille...",Ethiopia
40804525,Joshua R Robinson,"[Archaeology Program, Boston University, Bosto...","Archaeology Program, Boston University, Boston...",USA
40804525,Eric Scott,"[Department of Biology, College of Natural Sci...","Department of Biology, College of Natural Scie...",USA
40804525,Irene E Smail,"[Department of Biomedical Sciences, West Virgi...","Department of Biomedical Sciences, West Virgin...",USA
40804525,Kebede Geleta Terefe,"[Techno Link College Burayu Campus, Shager Cit...","Techno Link College Burayu Campus, Shager City...",Ethiopia
40804525,Lars Werdelin,"[Department of Palaeobiology, Swedish Museum o...","Department of Palaeobiology, Swedish Museum of...",Sweden
40804525,William H Kimbel,[Institute of Human Origins and School of Huma...,Institute of Human Origins and School of Human...,USA
40804525,J Ramón Arrowsmith,[School of Earth and Space Exploration and Ins...,School of Earth and Space Exploration and Inst...,USA
40804525,Kaye E Reed,[Institute of Human Origins and School of Huma...,Institute of Human Origins and School of Human...,USA


In [328]:
# pmid_grouped = cell_author_info.groupby(cell_author_info.index)
# pmid_grouped['countries_involved'] = pmid_grouped['primary_country'].unique()

cell_author_info['primary_country'].value_counts()

primary_country
No affiliation    115524
USA               110124
UK                 23599
China              20379
Germany            19145
                   ...  
Nature Index          11
French Guiana         11
Jamaica               11
Namibia               11
Malawi                11
Name: count, Length: 162, dtype: int64

In [329]:
#countries_involved = cell_author_info.groupby(cell_author_info.index)['primary_country'].value_counts()
countries_involved = cell_author_info.groupby(['PMID', 'primary_country']).size()
countries_involved = pd.Series({
    pmid: group.droplevel(0).to_dict()
    for pmid, group in countries_involved.groupby(level=0)
})
#countries_involved = countries_involved.groupby(level=0).apply(lambda x: x.droplevel(0).to_dict())

In [330]:
countries_involved

10638731                       {'No affiliation': 1, 'UK': 1}
10638732                                            {'UK': 1}
10638733                                {'No affiliation': 2}
10638736                                {'No affiliation': 1}
10638737                                {'No affiliation': 1}
                                  ...                        
42310461    {'China': 33, 'Switzerland': 1, 'UK': 1, 'USA'...
42310462                                         {'China': 9}
42310463                                          {'USA': 20}
42310464    {'Japan': 4, 'No affiliation': 1, 'Pakistan': ...
42310465                   {'Japan': 3, 'UK': 28, 'USA': 162}
Length: 32917, dtype: object

In [331]:
'Maryland' in 'Maryland 20892'

True